In [ ]:
import pandas as pd
import sys
from pathlib import Path
sys.path.append("../src")

PATH_NAME = Path("../data/processed/spaceship_titanic_cleaned.csv")
df = pd.read_csv(PATH_NAME)
df.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Surname,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Santantines,True


In [ ]:
# split cabin column
df[["Deck", "Cabin number", "Side of ship"]] = df.Cabin.str.split("/", expand=True)
df.drop("Cabin", inplace=True, axis=1)

# fix cabin number and group number data types
df["Cabin number"] = df["Cabin number"].fillna(-1).astype(int)

In [62]:
# split passenger id into group number and passengers in the group
df[["Group number", "Passengers within group"]] = df.PassengerId.str.split("_", expand=True).astype(int)
df.drop("PassengerId", axis=1, inplace=True)
df["Group number"] = df["Group number"].astype(int)

In [64]:
# total spend
df["Total spend"] = (df.RoomService + df.FoodCourt + df.ShoppingMall + df.Spa + df.VRDeck).astype(float)

In [ ]:
# group size
df["Group size"] = df.groupby("Group number")["Group number"].transform("count")
# is alone
df["Is alone"] = (df["Group size"] == 1).astype(bool)

In [66]:
# no spending flag
df["No spending"] = (df["Total spend"] == 0).astype(bool)

In [67]:
# luxury spending 
df["Luxury spending"] = df.Spa + df.VRDeck

In [69]:
# basic spending
df["Basic spending"] = df.FoodCourt + df.ShoppingMall

In [70]:
# spending per person
df["Spending per person"] = df["Total spend"] / df["Group size"]

In [72]:
# move target to end
col_to_move = df.pop("Transported")
df.insert(len(df.columns), "Transported", col_to_move)

In [ ]:
# look at the final df
df.head()

,HomePlanet,CryoSleep,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,...,Group number,Passengers within group,Total spend,Group size,Is alone,No spending,Luxury spending,Basic spending,Spending per person,Transported
0,Europa,False,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,...,1,1,0.0,1,True,True,0.0,0.0,0.0,False
1,Earth,False,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,...,2,1,736.0,1,True,False,593.0,34.0,736.0,True
2,Europa,False,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,...,3,1,10383.0,2,False,False,6764.0,3576.0,5191.5,False
3,Europa,False,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,...,3,2,5176.0,2,False,False,3522.0,1654.0,2588.0,False
4,Earth,False,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,...,4,1,1091.0,1,True,False,567.0,221.0,1091.0,True


In [74]:
# save dataset
df.to_csv("../data/processed/spaceship_titanic_feature_engineered.csv", index=False)